<a href="https://colab.research.google.com/github/ldfha/RotemAI/blob/main/projects/pro14YOLO/yolo12tracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Ultralytics YOLO를 이용한 다중 객체 추적
# Ultralytics YOLO 기반 트래킹(추적)은 영상에서 객체를 탐지 + ID를 붙여 물체를 계속 추적하는 기술
# 2단계로 작업
# - Detection : 사람 / 동물 / 자동차 등 감지
# - Tracking : 같은 객체에 id를 부여해 Frame이 바뀌어도 동일 객체 유지
# 내부 알고리즘으로 ByteTrack, Bot-sort 등이 사용됨
!pip install ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.7 MB/s eta 0:00:00


In [2]:
from numpy import show_config
import cv2
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

video_path = "road_car.mp4"

cap = cv2.VideoCapture(video_path)  # 지정한 동영상 파일을 프레임 단위로 읽기

if not cap.isOpened():
  print('동영상 열기 실패')
  exit()

while True:
  ret, frame = cap.read()   # ret : 읽기 성공 여부, frame : 실제 이미지 데이터

  if not ret:
    break

  results = model.track(
      source=frame,   # 분석할 입력 이미지(현재 읽은 1장의 프레임)
      persist=True,
      tracker='bytetrack.yaml',   # 객체 추적 알고리즘 중 하나. 현재 프레임과 이전 프레임을 비교해 id를 부여
      verbose=False,
      show=False
  )

  result=results[0]
  if result.boxes is not None and result.boxes.id is not None:
    boxes = result.boxes.xyxy.cpu().numpy()
    ids = result.boxes.id.cpu().numpy().astype(int)
    class_ids = result.boxes.cls.cpu().numpy().astype(int)  # ex) car = 2

    # 박스 좌표, 추적 id, 클래스 번호를 묶어 반복 처리
    for box, track_id, class_id in zip(boxes, ids, class_ids):
      x1, y1, x2, y2 = map(int, box)
      class_name = result.names[class_id]   # ex) 2 -> car
      print(f'id : {track_id}, class : {class_name}, box :', x1, y1, x2, y2)

  annotated_frame = result.plot()   # YOLO가 탐지한 이미지 자동으로 그리기 (각종 정보가 표시됨)
  display_frame = cv2.resize(annotated_frame, (960, 540))

  cv2.imshow("YOLOv8 Tracking", display_frame)

  if cv2.waitKey(1) & 0xFF == ord('q'):   # q카 누르면 종료
    break

cap.release()
cv2.destroyAllWindows()

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
id : 7, class : car, box : 586 574 830 784
id : 8, class : car, box : 1040 1191 1576 1661
id : 11, class : car, box : 1248 1036 1699 1465
id : 40, class : car, box : 2348 1855 2984 2154
id : 59, class : car, box : 400 582 649 826
id : 91, class : car, box : 244 635 514 864
id : 96, class : car, box : 1411 647 1743 945
id : 25, class : car, box : 2261 1089 2742 1497
id : 10, class : truck, box : 2218 518 2616 916
id : 182, class : car, box : 27 670 304 914
id : 154, class : truck, box : 1651 445 1889 637
id : 233, class : car, box : 2094 852 2413 1163
id : 246, class : car, box : 16 766 110 1058
id : 156, class : car, box : 2217 892 2716 1225
id : 220, class : car, box : 2260 897 2676 1130
id : 242, class : car, box : 1292 831 1589 1037
id : 5, class : car, box : 2056 1308 2674 1834
id : 6, class : car, box : 793 1883 1449 2143
id : 7, class : car, box : 592 574 833 782
id : 8, class : car, box : 1037 1197 1572 1666
id : 11, class : car, box : 1247 10

KeyboardInterrupt: 